In [2]:
pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 44.2 MB/s eta 0:00:00


In [3]:
import faiss
import numpy as np
import nltk
from transformers import pipeline
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sentence_transformers import SentenceTransformer

nltk.download('punkt_tab')
nltk.download('stopwords')


class SimpleRAG:

    def __init__(self):
        # Load embedding model
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.stop_words = set(stopwords.words('english'))
        self.index = None
        self.documents = []

    # Step 1 & 2: Preprocess Data (Remove Stopwords)
    def preprocess(self, text):
        words = word_tokenize(text)
        filtered = [w for w in words if w.lower() not in self.stop_words]
        return " ".join(filtered)

    # Step 3: Convert to Embeddings
    def embed(self, texts):
        return self.model.encode(texts)

    # Step 4: Store in Vector DB
    def store_documents(self, documents):
        self.documents = [self.preprocess(doc) for doc in documents]
        embeddings = self.embed(self.documents)

        dimension = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dimension)
        self.index.add(np.array(embeddings))

    # Step 5: Search Similar Documents
    def search(self, query, k=2):
        processed_query = self.preprocess(query)
        query_vector = self.embed([processed_query])

        distances, indices = self.index.search(query_vector, k)
        results = [self.documents[i] for i in indices[0]]
        return results

    # Step 6: Generate Answer (Simple LLM Simulation)
    def generate_answer(self, query):
        generator = pipeline('text-generation',model='gpt2')
        retrieved_docs = self.search(query)

        context = " ".join(retrieved_docs)

      # Simple generation (replace with real LLM if needed)
        answer = f"Based on the documents, the answer is:\n{context}"
        # Generate answer
        result = generator(answer, max_length=150, num_return_sequences=1)
        return result
# Sample Data
documents = [
    "Machine Learning (ML) is a subfield of Artificial Intelligence concerned with designing algorithms that enable systems to learn patterns from data and make decisions or predictions without being explicitly programmed with rule-based logic. Instead of hard-coding instructions, ML models infer functional relationships between input features (independent variables) and target outputs (dependent variables) by optimizing an objective function over training data. At its core, ML is about **statistical estimation, optimization, and generalization**. A model is trained on historical data using an algorithm (e.g., gradient descent), where parameters are iteratively adjusted to minimize a loss function such as Mean Squared Error (regression) or Cross-Entropy (classification). The ultimate goal is not just memorizing training data but achieving strong performance on unseen data — this ability is called generalization.",
    "Machine Learning is broadly categorized into **Supervised Learning**, **Unsupervised Learning**, and **Reinforcement Learning**. In supervised learning, labeled datasets are used to train models for tasks such as regression (predicting continuous values like house prices) and classification (predicting categories like spam vs non-spam). Algorithms such as Linear Regression, Logistic Regression, Decision Trees, Random Forests, Support Vector Machines, and Neural Networks are commonly used. In unsupervised learning, the data has no labeled target variable, and the goal is to uncover hidden structures or patterns, such as clustering customers using K-Means or reducing dimensionality using PCA. Reinforcement learning, on the other hand, focuses on agents interacting with an environment, learning optimal strategies through reward-based feedback mechanisms, widely applied in robotics and game AI.",
    "A typical ML pipeline includes data collection, data preprocessing (handling missing values, encoding categorical variables, scaling features), feature engineering, model selection, hyperparameter tuning (e.g., using GridSearchCV), model evaluation (using metrics like Accuracy, F1-score, ROC-AUC for classification or RMSE and R² for regression), and deployment. Overfitting and underfitting are key concerns; regularization techniques such as L1 (Lasso) and L2 (Ridge) penalties help control model complexity. Cross-validation is used to ensure robust evaluation. In production systems, considerations expand to model drift, scalability, monitoring, and reproducibility.",
    "Modern ML has evolved significantly with the advancement of Deep Learning, where multi-layer neural networks automatically learn hierarchical feature representations from large-scale datasets. Applications of ML span across industries — recommendation systems (e-commerce), fraud detection (finance), predictive maintenance (manufacturing), NLP (chatbots, sentiment analysis), computer vision (image recognition), healthcare diagnostics, and autonomous vehicles. Ultimately, Machine Learning transforms raw data into actionable intelligence by leveraging mathematical modeling, probability theory, and computational efficiency to automate decision-making at scale."
]

# Create object
rag = SimpleRAG()

# Store data
rag.store_documents(documents)

# User Query
query = "What is artificial intelligence?"

# Generate Answer
print(rag.generate_answer(query))

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_length', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Based on the documents, the answer is:\nMachine Learning ( ML ) subfield Artificial Intelligence concerned designing algorithms enable systems learn patterns data make decisions predictions without explicitly programmed rule-based logic . Instead hard-coding instructions , ML models infer functional relationships input features ( independent variables ) target outputs ( dependent variables ) optimizing objective function training data . core , ML * * statistical estimation , optimization , generalization * * . model trained historical data using algorithm ( e.g. , gradient descent ) , parameters iteratively adjusted minimize loss function Mean Squared Error ( regression ) Cross-Entropy ( classification ) . ultimate goal memorizing training data achieving strong performance unseen data — ability called generalization . Modern ML evolved significantly advancement Deep Learning , multi-layer neural networks automatically learn hierarchical feature representations larg

In [4]:
import faiss
import numpy as np
import nltk
from transformers import pipeline
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sentence_transformers import SentenceTransformer

# Download required NLTK data
nltk.download('punkt_tab')
nltk.download('stopwords')


class SimpleRAG:

    def __init__(self):
        # Load embedding model
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.stop_words = set(stopwords.words('english'))
        self.index = None
        self.documents = []

    # Step 1 & 2: Preprocess Data (Remove Stopwords)
    def preprocess(self, text):
        words = word_tokenize(text)
        filtered = [w for w in words if w.lower() not in self.stop_words]
        return " ".join(filtered)

    # Step 3: Convert to Embeddings
    def embed(self, texts):
        return self.model.encode(texts)

    # Step 4: Store in Vector DB
    def store_documents(self, documents):
        self.documents = [self.preprocess(doc) for doc in documents]
        embeddings = self.embed(self.documents)

        dimension = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dimension)
        self.index.add(np.array(embeddings))

    # Step 5: Search Similar Documents
    def search(self, query, k=2):
        processed_query = self.preprocess(query)
        query_vector = self.embed([processed_query])

        distances, indices = self.index.search(query_vector, k)
        results = [self.documents[i] for i in indices[0]]
        return results

    # Step 6: Generate Answer (LLM Generation)
    def generate_answer(self, query):
        generator = pipeline('text-generation', model='gpt2')
        retrieved_docs = self.search(query)

        context = " ".join(retrieved_docs)

        # Proper Prompt Formatting
        prompt = f"""
Use the following context to answer the question.

Context:
{context}

Question:
{query}

Answer:
"""

        result = generator(
            prompt,
            max_length=300,
            num_return_sequences=1,
            truncation=True
        )

        return result[0]['generated_text']


# ------------------------------
# Sample Data
# ------------------------------

documents = [
    "Machine Learning (ML) is a subfield of Artificial Intelligence concerned with designing algorithms that enable systems to learn patterns from data and make decisions or predictions without being explicitly programmed with rule-based logic. Instead of hard-coding instructions, ML models infer functional relationships between input features (independent variables) and target outputs (dependent variables) by optimizing an objective function over training data. At its core, ML is about statistical estimation, optimization, and generalization.",

    "Machine Learning is broadly categorized into Supervised Learning, Unsupervised Learning, and Reinforcement Learning. In supervised learning, labeled datasets are used to train models for tasks such as regression and classification. In unsupervised learning, the data has no labeled target variable, and the goal is to uncover hidden structures or patterns. Reinforcement learning focuses on agents interacting with an environment and learning through reward-based feedback.",

    "A typical ML pipeline includes data collection, preprocessing, feature engineering, model selection, hyperparameter tuning, model evaluation, and deployment. Overfitting and underfitting are key concerns. Regularization techniques such as L1 and L2 help control model complexity.",

    "Modern ML has evolved significantly with Deep Learning, where multi-layer neural networks learn hierarchical feature representations. Applications include recommendation systems, fraud detection, NLP, computer vision, healthcare diagnostics, and autonomous vehicles."
]

# ------------------------------
# Run RAG
# ------------------------------

rag = SimpleRAG()

# Store data
rag.store_documents(documents)

# User Query
query = "What is deep learning?"

# Generate Answer
print(rag.generate_answer(query))

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=300) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Use the following context to answer the question.

Context:
Modern ML evolved significantly Deep Learning , multi-layer neural networks learn hierarchical feature representations . Applications include recommendation systems , fraud detection , NLP , computer vision , healthcare diagnostics , autonomous vehicles . Machine Learning ( ML ) subfield Artificial Intelligence concerned designing algorithms enable systems learn patterns data make decisions predictions without explicitly programmed rule-based logic . Instead hard-coding instructions , ML models infer functional relationships input features ( independent variables ) target outputs ( dependent variables ) optimizing objective function training data . core , ML statistical estimation , optimization , generalization .

Question:
What is deep learning?

Answer:

Deep learning is a data-driven, scalable, and scalable way of learning, and as such is the most powerful and scalable system built on top of Machine learning.

Deep Learni